# Combined Species Classifier — EfficientNet-B0

Image classification model using transfer learning with **EfficientNet-B0** on a merged dataset combining La Jolla Cove (~12.5k images, 50 classes) and UCSD campus (~4.9k images, 51 classes) observations.

**Training strategy:**
- Phase 1: Freeze backbone, train classification head only (higher LR)
- Phase 2: Unfreeze backbone, fine-tune all layers (lower LR)

## 1. Setup and Imports

In [ ]:
import os
import json
import copy
import time
import random
import shutil
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.amp import GradScaler, autocast

import torchvision
from torchvision import datasets, transforms, models

from sklearn.metrics import classification_report, confusion_matrix, top_k_accuracy_score
from PIL import Image

print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    print("MPS (Apple Silicon) available")

### Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
LJC_DATA = Path("../species_identification/data")
UCSD_DATA = Path("../ucsd_species/data")
MERGED_DATA = Path("data")             # merged symlink tree
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Model ──────────────────────────────────────────────────────────────────────
IMAGE_SIZE = 224

# ── Phase 1: Head-only training ────────────────────────────────────────────────
P1_EPOCHS = 10
P1_LR = 1e-3
P1_BATCH_SIZE = 64

# ── Phase 2: Full fine-tuning ──────────────────────────────────────────────────
P2_EPOCHS = 20
P2_LR = 1e-5
P2_BATCH_SIZE = 32

EARLY_STOP_PATIENCE = 5

# ── Hardware ───────────────────────────────────────────────────────────────────
NUM_WORKERS = 4
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
USE_AMP = DEVICE.type == "cuda"

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Device:     {DEVICE}")
print(f"AMP:        {USE_AMP}")
print(f"Phase 1:    {P1_EPOCHS} epochs, lr={P1_LR}, batch={P1_BATCH_SIZE}")
print(f"Phase 2:    {P2_EPOCHS} epochs, lr={P2_LR}, batch={P2_BATCH_SIZE}")

## 2. Merge Datasets

Create a unified `data/train/val/test` tree by symlinking images from both source datasets. Overlapping species (same folder name) get their images combined.

In [ ]:
def merge_datasets(sources, dest, splits=("train", "val", "test")):
    """
    Merge multiple ImageFolder datasets via symlinks.
    Overlapping class folders get their images combined.
    """
    if dest.exists():
        shutil.rmtree(dest)

    stats = {"sources": len(sources), "splits": {}, "total_images": 0, "total_classes": 0}

    all_classes = set()
    for split in splits:
        split_dir = dest / split
        split_images = 0
        split_classes = set()

        for src in sources:
            src_split = src / split
            if not src_split.exists():
                print(f"  Warning: {src_split} does not exist, skipping")
                continue

            for class_dir in sorted(src_split.iterdir()):
                if not class_dir.is_dir():
                    continue
                cls_name = class_dir.name
                split_classes.add(cls_name)
                all_classes.add(cls_name)

                dest_cls = split_dir / cls_name
                dest_cls.mkdir(parents=True, exist_ok=True)

                for img in class_dir.iterdir():
                    if img.suffix.lower() in (".jpg", ".jpeg", ".png"):
                        link = dest_cls / img.name
                        if not link.exists():
                            link.symlink_to(img.resolve())
                            split_images += 1

        stats["splits"][split] = {"images": split_images, "classes": len(split_classes)}
        stats["total_images"] += split_images

    stats["total_classes"] = len(all_classes)
    return stats


print("Merging La Jolla Cove + UCSD datasets...")
merge_stats = merge_datasets([LJC_DATA, UCSD_DATA], MERGED_DATA)

print(f"\nMerged dataset: {merge_stats['total_classes']} classes, {merge_stats['total_images']} images")
for split, info in merge_stats["splits"].items():
    print(f"  {split:>5s}: {info['classes']} classes, {info['images']} images")

## 3. Dataset Loading and Inspection

In [ ]:
_inspect_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

train_dataset_raw = datasets.ImageFolder(MERGED_DATA / "train", transform=_inspect_tf)
val_dataset_raw   = datasets.ImageFolder(MERGED_DATA / "val",   transform=_inspect_tf)
test_dataset_raw  = datasets.ImageFolder(MERGED_DATA / "test",  transform=_inspect_tf)

class_names = train_dataset_raw.classes
num_classes = len(class_names)

print(f"Number of classes: {num_classes}")
print(f"Class names: {class_names}\n")

for split_name, ds in [("Train", train_dataset_raw), ("Val", val_dataset_raw), ("Test", test_dataset_raw)]:
    targets = [s[1] for s in ds.samples]
    counts = Counter(targets)
    print(f"  {split_name}: {len(ds)} images")
    for idx in sorted(counts):
        print(f"    {ds.classes[idx]:>50s}: {counts[idx]}")
    print()

### Sample images

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
indices = random.sample(range(len(train_dataset_raw)), 10)
for ax, idx in zip(axes.flat, indices):
    img, label = train_dataset_raw[idx]
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(class_names[label].replace("_", " "), fontsize=8)
    ax.axis("off")
fig.suptitle("Sample Training Images (Combined Dataset)", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Transforms and DataLoaders

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_test_transforms = transforms.Compose([
    transforms.Resize(int(IMAGE_SIZE * 1.14)),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

In [ ]:
def make_loaders(batch_size):
    """Create DataLoaders with a given batch size (changes between phases)."""
    train_ds = datasets.ImageFolder(MERGED_DATA / "train", transform=train_transforms)
    val_ds   = datasets.ImageFolder(MERGED_DATA / "val",   transform=val_test_transforms)
    test_ds  = datasets.ImageFolder(MERGED_DATA / "test",  transform=val_test_transforms)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    return train_ds, val_ds, test_ds, train_loader, val_loader, test_loader


print("make_loaders() defined.")

## 5. Model

EfficientNet-B0 pretrained on ImageNet with a custom classification head.

In [ ]:
def build_efficientnet(num_classes, freeze_backbone=True):
    """Build EfficientNet-B0 with a fresh classification head."""
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

    if freeze_backbone:
        for param in model.features.parameters():
            param.requires_grad = False

    in_features = model.classifier[1].in_features  # 1280 for B0
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, num_classes),
    )
    return model


model = build_efficientnet(num_classes, freeze_backbone=True)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:>12,}")
print(f"Trainable parameters: {trainable_params:>12,}")
print(f"Frozen parameters:    {total_params - trainable_params:>12,}")

## 6. Training Utilities

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler, device, use_amp):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast("cuda", enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device, use_amp):
    model.eval()
    running_loss = 0.0
    correct_top1 = 0
    correct_top3 = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        with autocast("cuda", enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        total += labels.size(0)

        _, preds_top1 = outputs.max(1)
        correct_top1 += preds_top1.eq(labels).sum().item()

        _, preds_top3 = outputs.topk(min(3, outputs.size(1)), dim=1)
        correct_top3 += preds_top3.eq(labels.unsqueeze(1)).any(1).sum().item()

    return running_loss / total, correct_top1 / total, correct_top3 / total


def run_training_phase(model, train_loader, val_loader, criterion, optimizer, scheduler,
                       scaler, device, use_amp, num_epochs, patience, phase_name,
                       best_val_acc=0.0, best_model_wts=None):
    """Run a full training phase with early stopping."""
    history = {
        "train_loss": [], "val_loss": [],
        "train_acc": [], "val_acc": [], "val_top3_acc": [],
    }
    epochs_no_improve = 0

    print(f"\n{'='*85}")
    print(f"  {phase_name}")
    print(f"{'='*85}")
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Trainable params: {trainable:,}")
    print(f"{'Epoch':>5}  {'Train Loss':>10}  {'Val Loss':>10}  {'Train Acc':>10}  {'Val Acc':>10}  {'Val Top3':>10}  {'LR':>10}  {'Time':>6}")
    print("-" * 85)

    for epoch in range(1, num_epochs + 1):
        t0 = time.time()

        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler, device, use_amp)
        val_loss, val_acc, val_top3 = evaluate(model, val_loader, criterion, device, use_amp)
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["val_top3_acc"].append(val_top3)

        elapsed = time.time() - t0
        current_lr = scheduler.get_last_lr()[0]

        marker = ""
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
            marker = " *"
        else:
            epochs_no_improve += 1

        print(f"{epoch:>5d}  {train_loss:>10.4f}  {val_loss:>10.4f}  {train_acc:>10.4f}  {val_acc:>10.4f}  {val_top3:>10.4f}  {current_lr:>10.2e}  {elapsed:>5.1f}s{marker}")

        if epochs_no_improve >= patience:
            print(f"\nEarly stopping at epoch {epoch} (no improvement for {patience} epochs)")
            break

    print(f"Best val accuracy after {phase_name}: {best_val_acc:.4f}")
    return history, best_val_acc, best_model_wts


print("Training utilities defined.")

## 7. Phase 1 — Train Classification Head

Backbone is frozen. Only the classifier head is trained with a higher learning rate.

In [ ]:
train_ds, val_ds, test_ds, train_loader, val_loader, test_loader = make_loaders(P1_BATCH_SIZE)
print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

In [ ]:
criterion = nn.CrossEntropyLoss()

p1_optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=P1_LR, weight_decay=1e-2,
)
p1_scheduler = optim.lr_scheduler.CosineAnnealingLR(p1_optimizer, T_max=P1_EPOCHS, eta_min=1e-6)
scaler = GradScaler("cuda", enabled=USE_AMP)

p1_history, best_val_acc, best_model_wts = run_training_phase(
    model, train_loader, val_loader, criterion,
    p1_optimizer, p1_scheduler, scaler,
    DEVICE, USE_AMP,
    num_epochs=P1_EPOCHS,
    patience=EARLY_STOP_PATIENCE,
    phase_name="Phase 1: Head Only",
)

## 8. Phase 2 — Fine-Tune Entire Network

Unfreeze the backbone and train all parameters with a much lower learning rate.

In [ ]:
# Restore best Phase 1 weights before unfreezing
model.load_state_dict(best_model_wts)

# Unfreeze backbone
for param in model.features.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"All parameters now trainable: {trainable:,}")

In [ ]:
# Rebuild loaders with smaller batch size for full fine-tuning
train_ds, val_ds, test_ds, train_loader, val_loader, test_loader = make_loaders(P2_BATCH_SIZE)

p2_optimizer = optim.AdamW(model.parameters(), lr=P2_LR, weight_decay=1e-2)
p2_scheduler = optim.lr_scheduler.CosineAnnealingLR(p2_optimizer, T_max=P2_EPOCHS, eta_min=1e-7)

p2_history, best_val_acc, best_model_wts = run_training_phase(
    model, train_loader, val_loader, criterion,
    p2_optimizer, p2_scheduler, scaler,
    DEVICE, USE_AMP,
    num_epochs=P2_EPOCHS,
    patience=EARLY_STOP_PATIENCE,
    phase_name="Phase 2: Full Fine-Tune",
    best_val_acc=best_val_acc,
    best_model_wts=best_model_wts,
)

### Training curves

In [ ]:
# Combine histories from both phases
history = {k: p1_history[k] + p2_history[k] for k in p1_history}
p1_len = len(p1_history["train_loss"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(history["train_loss"]) + 1)

ax1.plot(epochs_range, history["train_loss"], label="Train")
ax1.plot(epochs_range, history["val_loss"], label="Val")
ax1.axvline(x=p1_len + 0.5, color="gray", linestyle="--", alpha=0.5, label="Phase 1 → 2")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history["train_acc"], label="Train")
ax2.plot(epochs_range, history["val_acc"], label="Val Top-1")
ax2.plot(epochs_range, history["val_top3_acc"], label="Val Top-3", linestyle="--")
ax2.axvline(x=p1_len + 0.5, color="gray", linestyle="--", alpha=0.5, label="Phase 1 → 2")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy")
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1.0))

fig.suptitle("Training Curves — EfficientNet-B0 (Two-Phase)", fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {OUTPUT_DIR / 'training_curves.png'}")

## 9. Evaluation on Test Set

In [ ]:
model.load_state_dict(best_model_wts)
model.eval()

test_loss, test_top1, test_top3 = evaluate(model, test_loader, criterion, DEVICE, USE_AMP)
print(f"Test Loss:         {test_loss:.4f}")
print(f"Test Top-1 Acc:    {test_top1:.4f}")
print(f"Test Top-3 Acc:    {test_top3:.4f}")

### Classification report and confusion matrix

In [ ]:
@torch.no_grad()
def collect_predictions(model, loader, device, use_amp):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        with autocast("cuda", enabled=use_amp):
            outputs = model(images)
        probs = torch.softmax(outputs.float(), dim=1)
        all_labels.append(labels)
        all_preds.append(outputs.argmax(1).cpu())
        all_probs.append(probs.cpu())

    return (torch.cat(all_labels).numpy(),
            torch.cat(all_preds).numpy(),
            torch.cat(all_probs).numpy())


y_true, y_pred, y_probs = collect_predictions(model, test_loader, DEVICE, USE_AMP)

print("Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig_size = max(12, num_classes * 0.35)
fig, ax = plt.subplots(figsize=(fig_size, fig_size))

sns.heatmap(cm, annot=num_classes <= 30, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names, ax=ax,
            linewidths=0.5, linecolor="white")
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("True", fontsize=12)
ax.set_title("Confusion Matrix — Test Set", fontsize=14)
plt.xticks(rotation=90, fontsize=6)
plt.yticks(rotation=0, fontsize=6)
plt.tight_layout()
plt.show()

if num_classes >= 3:
    sk_top3 = top_k_accuracy_score(y_true, y_probs, k=3, labels=range(num_classes))
    print(f"sklearn top-3 accuracy (verification): {sk_top3:.4f}")

## 10. Prediction on a Single Image

In [ ]:
@torch.no_grad()
def predict_single_image(image_path, model, transform, class_names, device, top_k=3):
    img = Image.open(image_path).convert("RGB")
    img_tensor = transform(img).unsqueeze(0).to(device)

    model.eval()
    with autocast("cuda", enabled=USE_AMP):
        logits = model(img_tensor)
    probs = torch.softmax(logits.float(), dim=1).squeeze(0)

    top_probs, top_indices = probs.topk(top_k)
    results = [(class_names[idx], prob.item()) for idx, prob in zip(top_indices, top_probs)]
    return img, results


sample_path, sample_label = test_ds.samples[random.randint(0, len(test_ds) - 1)]
true_name = class_names[sample_label]

img, predictions = predict_single_image(sample_path, model, val_test_transforms, class_names, DEVICE, top_k=3)

fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1, 1.5]})

ax_img.imshow(img)
ax_img.set_title(f"True: {true_name}", fontsize=12)
ax_img.axis("off")

names = [p[0] for p in predictions]
probs_vals = [p[1] for p in predictions]
colors = ["#2ecc71" if n == true_name else "#3498db" for n in names]
ax_bar.barh(range(len(names)), probs_vals, color=colors)
ax_bar.set_yticks(range(len(names)))
ax_bar.set_yticklabels(names, fontsize=10)
ax_bar.set_xlabel("Probability")
ax_bar.set_title("Top-3 Predictions")
ax_bar.set_xlim(0, 1)
ax_bar.invert_yaxis()

plt.tight_layout()
plt.show()

for rank, (name, prob) in enumerate(predictions, 1):
    print(f"  #{rank}: {name:<45s} {prob:.4f}")

## 11. Saving Artifacts

In [ ]:
# ── 1. Best model checkpoint ───────────────────────────────────────────────────
checkpoint_path = OUTPUT_DIR / "best_model.pth"
torch.save({
    "model_state_dict": best_model_wts,
    "model_name": "efficientnet_b0",
    "num_classes": num_classes,
    "class_names": class_names,
    "image_size": IMAGE_SIZE,
    "best_val_acc": best_val_acc,
    "phase1_epochs": len(p1_history["train_loss"]),
    "phase2_epochs": len(p2_history["train_loss"]),
}, checkpoint_path)
print(f"Saved model checkpoint: {checkpoint_path}")

# ── 2. Class names ─────────────────────────────────────────────────────────────
class_names_path = OUTPUT_DIR / "class_names.json"
with open(class_names_path, "w") as f:
    json.dump(class_names, f, indent=2)
print(f"Saved class names:     {class_names_path}")

# ── 3. Metrics ─────────────────────────────────────────────────────────────────
metrics = {
    "best_val_acc": best_val_acc,
    "test_top1_acc": test_top1,
    "test_top3_acc": test_top3,
    "test_loss": test_loss,
    "model_name": "efficientnet_b0",
    "image_size": IMAGE_SIZE,
    "phase1": {
        "epochs": len(p1_history["train_loss"]),
        "batch_size": P1_BATCH_SIZE,
        "learning_rate": P1_LR,
    },
    "phase2": {
        "epochs": len(p2_history["train_loss"]),
        "batch_size": P2_BATCH_SIZE,
        "learning_rate": P2_LR,
    },
    "num_classes": num_classes,
    "datasets": ["la_jolla_cove", "ucsd"],
    "history": history,
}
metrics_path = OUTPUT_DIR / "metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Saved metrics:         {metrics_path}")

print(f"Training curves:       {OUTPUT_DIR / 'training_curves.png'}")
print("\nAll artifacts saved.")

## Summary

Artifacts saved to `outputs/`:
- `best_model.pth` — EfficientNet-B0 checkpoint (state dict + metadata)
- `class_names.json` — ordered list of all species class names
- `metrics.json` — training/evaluation metrics and full history
- `training_curves.png` — loss and accuracy plots

**Model:** EfficientNet-B0 trained in two phases on the combined La Jolla Cove + UCSD dataset.

**Next steps:**
- Compare results with the DINOv2 models
- Try EfficientNet-B2 for potentially higher accuracy
- Connect to the YOLO detector pipeline for end-to-end inference